In [ ]:
import re
from pathlib import Path

import pandas as pd
import spiceypy as spice

from imap_processing.cdf.utils import load_cdf, write_cdf
from imap_processing.spice.time import et_to_ttj2000ns, et_to_utc, ttj2000ns_to_et

# Load in correct order:
kernels = [
    "/Users/luco3133/projects/imap_processing/data/imap/spice/lsk/naif016.tls",
    # Leap seconds
    "/Users/luco3133/projects/imap_processing/data/imap/spice/sclk/imap_sclk_005.tsc",
]

for kernel in kernels:
    print(f"Loading: {kernel}")
    spice.furnsh(kernel)

# Your directories
cdf_dir = Path("/Users/luco3133/projects/imap_processing/data/imap/ultra/l1c/2025")
spin_dir = Path(
    "/Users/luco3133/projects/ultra_stuff/validation_stuff/final_20251009/ultra-45-input"
)

# First, delete any existing v101 files
print("=" * 70)
print("DELETING EXISTING v101 FILES")
print("=" * 70 + "\n")

# v101_files = list(cdf_dir.glob('*repoint*v001.cdf'))
# print(f"Found {len(v101_files)} v101 files to delete")
#
# for v101_file in v101_files:
#     print(f"Deleting: {v101_file.name}")
#     v101_file.unlink()
#
# print(f"\n✓ Deleted {len(v101_files)} v101 files\n")

# Get only CDF files matching the pattern with repoint and v100
cdf_files = sorted(cdf_dir.glob("*/imap_ultra_l1c_45sensor-spacecraftpset_*_v001.cdf"))
print(cdf_files)
# Get only SpinTable files
spin_files = sorted(spin_dir.glob("SpinTable-p*.csv"))

print("=" * 70)
print("PROCESSING v100 FILES")
print("=" * 70 + "\n")
print(f"Found {len(cdf_files)} CDF files with repoint*_v100 pattern")
print(f"Found {len(spin_files)} SpinTable files\n")

# Build a dictionary mapping pointing number to spin file first time
spin_times = {}
for spin_file in spin_files:
    match = re.search(r"SpinTable-p(\d+)", spin_file.name)
    pointing_num = int(match.group(1))
    df = pd.read_csv(spin_file)
    first_time_et = df["Spin Start (tdb)"][0]

    # Convert ET (ephemeris time in seconds) to TT2000 nanoseconds
    first_time_ns = et_to_ttj2000ns(first_time_et)

    spin_times[pointing_num] = first_time_ns
    print(et_to_utc(ttj2000ns_to_et(first_time_et)))
    print(et_to_utc(first_time_et))
    print(f"Pointing {pointing_num}: first time = {first_time_et} ET")

print("\n" + "=" * 70)
print("Processing CDF files...")
print("=" * 70 + "\n")

pointing_num = 0
# Process each CDF file
for cdf_file in cdf_files:
    # match = re.search(r'repoint(\d+)', cdf_file.name)
    # pointing_num = int(match.group(1))

    print(f"Processing: {cdf_file.name}")
    print(f"  Pointing: {pointing_num}")

    ds = load_cdf(str(cdf_file))

    print(f"  Old epoch: {ds['epoch'].values}")
    print(pointing_num, cdf_file.name)
    if pointing_num == 0:
        curr_time = spin_times[pointing_num]
        # Subtract 1 hour (in nanoseconds)
        new_epoch = curr_time - int(3600 * 1e9)
    else:
        prev_time = spin_times[pointing_num - 1]
        curr_time = spin_times[pointing_num]
        # Midpoint of gap (both already in nanoseconds)
        new_epoch = (prev_time + curr_time) // 2
    print(et_to_utc(ttj2000ns_to_et(new_epoch)))
    ds["epoch"].values[0] = new_epoch

    print(f"  New epoch: {ds['epoch'].values}")

    ds.attrs["Data_version"] = "002"
    print("  Updated Data_version to: v101")

    write_cdf(ds)
    pointing_num += 1
    print("  ✓ Written\n")

print("=" * 70)
print("Done!")
print("=" * 70)

In [10]:
from zipfile import ZipFile

cdf_dir = Path("/Users/luco3133/projects/imap_processing/data/imap/ultra/l1c/2025")

# Get only CDF files matching the pattern with repoint and v100
cdf_files = sorted(cdf_dir.glob("*/imap_ultra_l1c_45sensor-spacecraftpset_*_v002.cdf"))
print([cdf.name for cdf in sorted(cdf_files)])
print(len(cdf_files))
with ZipFile("ultra_45_6mo_psets", "w") as zip:
    # writing each file one by one
    for file in cdf_files:
        zip.write(file)

['imap_ultra_l1c_45sensor-spacecraftpset_20250416_v002.cdf', 'imap_ultra_l1c_45sensor-spacecraftpset_20250417_v002.cdf', 'imap_ultra_l1c_45sensor-spacecraftpset_20250418_v002.cdf', 'imap_ultra_l1c_45sensor-spacecraftpset_20250419_v002.cdf', 'imap_ultra_l1c_45sensor-spacecraftpset_20250420_v002.cdf', 'imap_ultra_l1c_45sensor-spacecraftpset_20250421_v002.cdf', 'imap_ultra_l1c_45sensor-spacecraftpset_20250422_v002.cdf', 'imap_ultra_l1c_45sensor-spacecraftpset_20250423_v002.cdf', 'imap_ultra_l1c_45sensor-spacecraftpset_20250424_v002.cdf', 'imap_ultra_l1c_45sensor-spacecraftpset_20250425_v002.cdf', 'imap_ultra_l1c_45sensor-spacecraftpset_20250426_v002.cdf', 'imap_ultra_l1c_45sensor-spacecraftpset_20250427_v002.cdf', 'imap_ultra_l1c_45sensor-spacecraftpset_20250428_v002.cdf', 'imap_ultra_l1c_45sensor-spacecraftpset_20250429_v002.cdf', 'imap_ultra_l1c_45sensor-spacecraftpset_20250430_v002.cdf', 'imap_ultra_l1c_45sensor-spacecraftpset_20250501_v002.cdf', 'imap_ultra_l1c_45sensor-spacecraftpset

In [ ]:
import datetime
import re
from pathlib import Path

import pandas as pd
import spiceypy as spice

from imap_processing.cdf.utils import load_cdf, write_cdf
from imap_processing.spice.time import et_to_ttj2000ns, et_to_utc, ttj2000ns_to_et

# Your directories
cdf_dir = Path("/Users/luco3133/projects/imap_processing/data/imap/ultra/l1c/2026/09")
# Get only CDF files matching the pattern with repoint and v100
cdf_files = sorted(cdf_dir.glob("*repoint*_v001.cdf"))


print("=" * 70)
print("PROCESSING v101 FILES")
print("=" * 70 + "\n")
print(f"Found {len(cdf_files)} CDF files with repoint*_v100 pattern")

file_start_date = datetime.datetime(2025, 4, 14)
# Process each CDF file
for cdf_file in cdf_files:
    ds = load_cdf(cdf_file)
    ds.attrs["Repointing"] = ""
    ds.attrs["Start_date"] = (file_start_date + datetime.timedelta(days=1)).strftime(
        "%Y%m%d"
    )
    ds.attrs["Data_version"] = "102"
    print("  Updated Data_version to: v102")

    file = write_cdf(ds)
    print(file.as_posix())
    file_start_date = file_start_date + datetime.timedelta(days=1)
    print("  ✓ Written\n")

print("=" * 70)
print("Done!")
print("=" * 70)